# 27 — Skills Catalog, Tool Bundles & Power Features

Use this notebook to explore the full SHIPIT skills catalog, see how skills auto-attach tools,
verify tool bundle integrity, and run agents with skills-powered workflows.

In [1]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from shipit_agent import Agent, DeepAgent, DEFAULT_SKILLS_PATH, FileSkillRegistry
from shipit_agent.builtins import get_builtin_tool_map
from shipit_agent.skills.tool_bundles import (
    SKILL_TOOL_BUNDLES,
    tool_names_for_skills,
    validate_tool_bundles,
)

registry = FileSkillRegistry(DEFAULT_SKILLS_PATH)
print(f"Catalog path: {DEFAULT_SKILLS_PATH}")
print(f"Total skills: {len(registry.list())}")

Catalog path: /Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent/shipit_agent/skills/skills.json
Total skills: 37


## 1. Browse All Skills

The packaged catalog ships with 37 skills across development, research, marketing, DevOps, and more.

In [2]:
all_skills = registry.list()

print(f"{'ID':35} {'Category':20} Name")
print("-" * 85)
for skill in all_skills:
    print(f"{skill.id:35} {(skill.category or 'uncategorized'):20} {skill.display_name or skill.name}")

ID                                  Category             Name
-------------------------------------------------------------------------------------
web-scraper-pro                     Web Scraping         Web Scraper Pro
portfolio-website-builder           utilities            Portfolio Website Builder
google-maps-lead-finder             Leads & Sales        Google Maps Lead Finder
amazon-affiliate-marketing-agent    Marketing            Amazon Affiliate Marketing Agent
gmail-and-calendar-agent            Productivity         Gmail and Calendar Agent
ugc-video-ad-agent                  Video                UGC Video Ad Agent
video-clone-agent                   Video                Video Clone Agent
ai-music-generator                  Audio                AI Music Generator
startup-idea-scout                  Research             Startup Idea Scout
lead-enrichment                     Sales & CRM          Lead Enrichment
competitor-intelligence-tracker     Data & APIs          Competitor

## 2. Search The Catalog

Use fuzzy search to find relevant prebuilt skills before attaching them to an agent.

In [3]:
for query in ["database", "web scrape", "security", "marketing", "devops"]:
    matches = registry.search(query)
    print(f"\n'{query}' → {len(matches)} match(es)")
    for skill in matches[:3]:
        print(f"  - {skill.id}: {skill.description[:80]}")


'database' → 4 match(es)
  - database-architect: Design schemas, optimize queries, choose indexes, plan migrations, and improve d
  - notion-workspace-manager: Helps you create, update, search, and organize pages and databases in Notion.
  - mcp-server-builder: Helps you build MCP servers with custom tools, integrations, and resources for A

'web scrape' → 2 match(es)
  - web-scraper-pro: Scrapes modern websites with JavaScript support, anti-bot handling, login sessio
  - security-engineer: Protect your applications and infrastructure with expert security analysis. Find

'security' → 4 match(es)
  - security-engineer: Protect your applications and infrastructure with expert security analysis. Find
  - api-test-assistant: Checks your API for broken endpoints, bad responses, slow requests, and common s
  - devops-automation: Automate your deployment pipeline. Set up CI/CD, containers, and infrastructure 

'marketing' → 7 match(es)
  - marketing-advisor: Get clear marketing advice, bette

## 3. Tool Bundles — Skills Auto-Attach Tools

Every packaged skill has a **tool bundle** — a list of built-in tools that the skill needs.
When a skill is selected, its tools are automatically injected into the agent's toolset.

This means skills shape both **how the agent thinks** and **which tools it can use**.

In [4]:
print(f"Total skills with tool bundles: {len(SKILL_TOOL_BUNDLES)}")
print()

# Show tool bundles for a few interesting skills
for skill_id in ["web-scraper-pro", "code-workflow-assistant", "security-engineer", "full-stack-developer"]:
    tools = SKILL_TOOL_BUNDLES.get(skill_id, [])
    print(f"{skill_id}:")
    print(f"  Tools ({len(tools)}): {', '.join(tools)}")
    print()

Total skills with tool bundles: 37

web-scraper-pro:
  Tools (10): web_search, open_url, playwright_browse, bash, glob_files, grep_files, read_file, edit_file, write_file, run_code

code-workflow-assistant:
  Tools (10): read_file, edit_file, write_file, glob_files, grep_files, bash, run_code, workspace_files, plan_task, verify_output

security-engineer:
  Tools (9): bash, web_search, open_url, playwright_browse, read_file, grep_files, glob_files, run_code, verify_output

full-stack-developer:
  Tools (13): read_file, edit_file, write_file, glob_files, grep_files, bash, run_code, workspace_files, web_search, open_url, playwright_browse, plan_task, verify_output



## 4. Validate Tool Bundles

Verify that every tool name in SKILL_TOOL_BUNDLES actually exists in the built-in tool map.
This catches tool name typos or stale references.

In [5]:
# Use a stub LLM so SubAgentTool is included
class StubLLM:
    def complete(self, **kwargs):
        pass

builtin_names = set(get_builtin_tool_map(llm=StubLLM()).keys())
print(f"Built-in tools available: {len(builtin_names)}")
print(f"Tool names: {sorted(builtin_names)}")
print()

errors = validate_tool_bundles(builtin_names)
if errors:
    print(f"ERRORS — {len(errors)} unknown tool references:")
    for err in errors:
        print(f"  - {err}")
else:
    print("All tool bundle references are valid.")

Built-in tools available: 32
Tool names: ['ask_user', 'bash', 'build_artifact', 'build_prompt', 'confluence', 'custom_api', 'decision_matrix', 'decompose_problem', 'edit_file', 'glob_files', 'gmail_search', 'google_calendar', 'google_drive', 'grep_files', 'human_review', 'jira', 'linear', 'memory', 'notion', 'open_url', 'plan_task', 'playwright_browse', 'read_file', 'run_code', 'slack', 'sub_agent', 'synthesize_evidence', 'tool_search', 'verify_output', 'web_search', 'workspace_files', 'write_file']

All tool bundle references are valid.


## 5. Build An Agent With Skills

Skills are passed with `skills=[...]`. The agent auto-attaches the skill's tool bundle.

When skills are active and `max_iterations` is at default (4), the runtime auto-boosts to 8
so the agent has enough turns to use the injected tools.

In [ ]:
from examples.run_multi_tool_agent import build_llm_from_env

llm = build_llm_from_env("bedrock")

agent = Agent.with_builtins(
    llm=llm,
    name="skills-demo-agent",
    skills=["database-architect", "code-workflow-assistant"],
    default_skill_ids=["brand-voice-guide"],
    auto_use_skills=True,
    skill_match_limit=2,
)

print("Attached skills:")
for skill in agent.skills:
    print(f"  - {skill.id}")

# Check iteration boost
selected = agent._selected_skills("test prompt")
effective_max = agent._effective_max_iterations(selected)
print(f"\nDefault max_iterations: {agent.max_iterations}")
print(f"Effective max_iterations (with skills): {effective_max}")

# Check injected tools
effective_tools = agent._effective_tools("test", selected_skills=selected)
tool_names = sorted({getattr(t, 'name', '') for t in effective_tools})
print(f"\nTotal effective tools: {len(effective_tools)}")
print(f"Tool names: {tool_names}")

## 6. Skill Tool Names For A Skill Set

Use `tool_names_for_skills()` to see what tools a combination of skills would inject.

In [ ]:
# Combine two skills and see the merged tool list
web_skill = registry.get("web-scraper-pro")
code_skill = registry.get("code-workflow-assistant")

combined_tools = tool_names_for_skills([web_skill, code_skill])
print(f"Tools from web-scraper-pro + code-workflow-assistant:")
for name in combined_tools:
    print(f"  - {name}")
print(f"\nTotal unique tools: {len(combined_tools)}")

## 7. Run The Agent

This run combines explicit attached skills with automatic skill matching from the prompt.
The agent gets more iterations and the right tools automatically.

In [ ]:
result = agent.run(
    "Plan this feature, debug the failing query path, and suggest database improvements."
)

print(f"Used skills: {result.metadata['used_skills']}")
print(f"Skill-injected tools: {result.metadata['used_skill_tools']}")
print()
display(Markdown(result.output))

## 8. Add Skills At Runtime

In [ ]:
added = agent.add_skill("security-engineer")
print(f"Added skill: {added.id}")

print("\nAll attached skills now:")
for skill in agent.skills:
    print(f"  - {skill.id}")

print("\nSearch from the live agent:")
for skill in agent.search_skills("devops")[:3]:
    print(f"  - {skill.id}: {skill.description[:60]}")

## 9. DeepAgent With Skills

`DeepAgent` supports the same skill API and passes it through to the inner `Agent`.
DeepAgent defaults to `max_iterations=8` already, so the boost mainly helps plain `Agent`.

In [ ]:
deep_agent = DeepAgent.with_builtins(
    llm=llm,
    name="skills-demo-deep-agent",
    skills=["database-architect", "full-stack-developer"],
    default_skill_ids=["code-workflow-assistant"],
    auto_use_skills=True,
)

print("DeepAgent attached skills:")
for skill in deep_agent.skills:
    print(f"  - {skill.id}")

print(f"\nInner agent tools: {len(deep_agent.tools)}")

print("\nDeepAgent catalog search ('marketing'):")
for skill in deep_agent.search_skills("marketing")[:3]:
    print(f"  - {skill.id}: {skill.description[:60]}")

In [ ]:
deep_result = deep_agent.run(
    "Investigate a slow database-backed feature, make a plan, and review the implementation approach."
)

display(Markdown(getattr(deep_result, "output", str(deep_result))))

## 10. Coverage Check — All Skills Have Bundles

Verify that every skill in the packaged catalog has a matching tool bundle.

In [ ]:
all_skills = registry.list()
missing = [s.id for s in all_skills if s.id not in SKILL_TOOL_BUNDLES]
covered = [s.id for s in all_skills if s.id in SKILL_TOOL_BUNDLES]

print(f"Skills with tool bundles: {len(covered)}/{len(all_skills)}")
if missing:
    print(f"Missing bundles: {missing}")
else:
    print("All skills have tool bundles.")

## Quick Reference

| Feature | API |
| --- | --- |
| Attach skills | `skills=["skill-id", ...]` |
| Default skill ids | `default_skill_ids=["skill-id", ...]` |
| Auto-match from prompt | `auto_use_skills=True` |
| Limit auto-matches | `skill_match_limit=3` |
| Add skill at runtime | `agent.add_skill("skill-id")` |
| Search catalog | `agent.search_skills("query")` |
| Check used skills | `result.metadata["used_skills"]` |
| Check injected tools | `result.metadata["used_skill_tools"]` |
| Validate bundles | `validate_tool_bundles(builtin_names)` |
| Get tool names for skills | `tool_names_for_skills([skill1, skill2])` |